<a href="https://colab.research.google.com/github/shivamkumar359/03-rd-SEM--ASSIGNMENTS/blob/main/IDB/W13/Q2/ASSIGNMNT_W13_Q_2_2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================================================
# TASK 2: IMPLEMENT A TWO-PHASE COMMIT (2PC) PROTOCOL WORKFLOW
# IIT Jodhpur BSc - Introduction to Databases
# ================================================================
#
# DELIVERABLES:
# 1. 2PC Sequence Diagram (PNG)
# 2. 2PC Sequence Diagram (PDF)
# 3. Transaction Log Sample (TXT)
# 4. Recovery Procedure (PDF)
# 5. Complete ZIP file
#
# ================================================================

# STEP 1: INSTALL REQUIRED LIBRARIES

!pip -q install reportlab pillow matplotlib


# STEP 2: IMPORT LIBRARIES

import os
import zipfile
from datetime import datetime

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle

from PIL import Image, ImageDraw, ImageFont

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    Image as PDFImage
)

from google.colab import files


# STEP 3: CREATE OUTPUT DIRECTORY

OUTPUT_DIR = "/content/Task_2_Two_Phase_Commit"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output directory created:", OUTPUT_DIR)


# ================================================================
# STEP 4: DEFINE 2PC PARTICIPANTS AND TRANSACTION
# ================================================================

TRANSACTION_ID = "TXN_2026_001"

COORDINATOR = "Coordinator"

PARTICIPANT_1 = "Site S1 - India"

PARTICIPANT_2 = "Site S2 - Singapore"

participants = [
    PARTICIPANT_1,
    PARTICIPANT_2
]

# Successful transaction scenario:
# Both participants vote READY.

participant_votes = {
    PARTICIPANT_1: "READY",
    PARTICIPANT_2: "READY"
}

print("\nTransaction ID:", TRANSACTION_ID)

print("Coordinator:", COORDINATOR)

print("Participants:", participants)


# ================================================================
# STEP 5: SIMULATE THE TWO-PHASE COMMIT WORKFLOW
# ================================================================

transaction_events = []

coordinator_state = "INITIAL"

participant_states = {
    PARTICIPANT_1: "INITIAL",
    PARTICIPANT_2: "INITIAL"
}


def record_event(site, event, state, detail):

    transaction_events.append({
        "site": site,
        "event": event,
        "state": state,
        "detail": detail
    })


# ---------------- PHASE 1: PREPARE ----------------

print("\n========== PHASE 1: PREPARE ==========")

coordinator_state = "PREPARING"

record_event(
    COORDINATOR,
    "BEGIN",
    coordinator_state,
    "Coordinator starts distributed transaction."
)

# Coordinator writes log before sending PREPARE

record_event(
    COORDINATOR,
    "GLOBAL_START",
    coordinator_state,
    "Transaction start recorded in stable storage."
)

for participant in participants:

    record_event(
        COORDINATOR,
        "SEND_PREPARE",
        coordinator_state,
        f"PREPARE message sent to {participant}."
    )

    # Participant receives PREPARE.
    # Participant writes PREPARED record to stable storage
    # BEFORE sending READY.

    participant_states[participant] = "PREPARED"

    record_event(
        participant,
        "PREPARE",
        "PREPARING",
        "Participant receives PREPARE request."
    )

    record_event(
        participant,
        "FORCE_PREPARED",
        "PREPARED",
        "PREPARED record written to stable storage."
    )

    # Vote response is sent only after stable log write.

    vote = participant_votes[participant]

    record_event(
        participant,
        f"SEND_{vote}",
        "PREPARED" if vote == "READY" else "ABORTED",
        f"Participant sends {vote} vote to coordinator."
    )

    record_event(
        COORDINATOR,
        f"RECEIVE_{vote}",
        coordinator_state,
        f"Coordinator receives {vote} from {participant}."
    )


# ---------------- PHASE 2: COMMIT OR ABORT ----------------

print("\n========== PHASE 2: DECISION ==========")

all_ready = all(
    vote == "READY"
    for vote in participant_votes.values()
)

if all_ready:

    global_decision = "COMMIT"

    coordinator_state = "COMMIT"

    # Coordinator MUST write COMMIT to stable storage
    # before sending COMMIT messages.

    record_event(
        COORDINATOR,
        "FORCE_GLOBAL_COMMIT",
        "COMMIT",
        "Global COMMIT decision written to stable storage."
    )

else:

    global_decision = "ABORT"

    coordinator_state = "ABORT"

    # Any single ABORT vote causes global ABORT.

    record_event(
        COORDINATOR,
        "FORCE_GLOBAL_ABORT",
        "ABORT",
        "Global ABORT decision written to stable storage."
    )


# Send final decision only after all votes are received.

for participant in participants:

    if global_decision == "COMMIT":

        record_event(
            COORDINATOR,
            "SEND_COMMIT",
            "COMMIT",
            f"COMMIT message sent to {participant}."
        )

        participant_states[participant] = "COMMITTED"

        record_event(
            participant,
            "RECEIVE_COMMIT",
            "COMMITTED",
            "Participant receives COMMIT decision."
        )

        record_event(
            participant,
            "FORCE_COMMIT",
            "COMMITTED",
            "COMMIT record written to stable storage."
        )

    else:

        record_event(
            COORDINATOR,
            "SEND_ABORT",
            "ABORT",
            f"ABORT message sent to {participant}."
        )

        participant_states[participant] = "ABORTED"

        record_event(
            participant,
            "RECEIVE_ABORT",
            "ABORTED",
            "Participant receives ABORT decision."
        )

        record_event(
            participant,
            "FORCE_ABORT",
            "ABORTED",
            "ABORT record written to stable storage."
        )


print("\n========== FINAL TRANSACTION STATUS ==========")

print("Global Decision:", global_decision)

print("Coordinator State:", coordinator_state)

for participant, state in participant_states.items():

    print(participant, ":", state)


# ================================================================
# STEP 6: CREATE TRANSACTION LOG SAMPLE (TXT)
# ================================================================

LOG_PATH = os.path.join(
    OUTPUT_DIR,
    "2PC_Transaction_Log.txt"
)

with open(LOG_PATH, "w", encoding="utf-8") as log:

    log.write("=" * 75 + "\n")
    log.write("DISTRIBUTED TRANSACTION LOG - TWO PHASE COMMIT\n")
    log.write("=" * 75 + "\n\n")

    log.write("Transaction ID: " + TRANSACTION_ID + "\n")

    log.write("Coordinator: " + COORDINATOR + "\n")

    log.write("Participants: Site S1 and Site S2\n\n")

    log.write("=" * 75 + "\n")
    log.write("PHASE 1: PREPARE PHASE\n")
    log.write("=" * 75 + "\n\n")

    log.write(
        "COORDINATOR LOG:\n"
        "[1] GLOBAL_START - Transaction initiated.\n"
        "[2] SEND_PREPARE - PREPARE sent to Site S1.\n"
        "[3] SEND_PREPARE - PREPARE sent to Site S2.\n\n"
    )

    log.write(
        "PARTICIPANT SITE S1 LOG:\n"
        "[4] PREPARE - Received PREPARE request.\n"
        "[5] FORCE_PREPARED - PREPARED record persisted.\n"
        "[6] SEND_READY - READY vote sent to coordinator.\n\n"
    )

    log.write(
        "PARTICIPANT SITE S2 LOG:\n"
        "[7] PREPARE - Received PREPARE request.\n"
        "[8] FORCE_PREPARED - PREPARED record persisted.\n"
        "[9] SEND_READY - READY vote sent to coordinator.\n\n"
    )

    log.write("=" * 75 + "\n")
    log.write("PHASE 2: COMMIT PHASE\n")
    log.write("=" * 75 + "\n\n")

    if global_decision == "COMMIT":

        log.write(
            "[10] COORDINATOR: Received READY from Site S1.\n"
            "[11] COORDINATOR: Received READY from Site S2.\n"
            "[12] FORCE_GLOBAL_COMMIT: COMMIT decision persisted.\n"
            "[13] SEND_COMMIT: COMMIT sent to Site S1.\n"
            "[14] SEND_COMMIT: COMMIT sent to Site S2.\n"
            "[15] SITE S1: COMMIT received and persisted.\n"
            "[16] SITE S2: COMMIT received and persisted.\n\n"
        )

    else:

        log.write(
            "[10] COORDINATOR: At least one participant voted ABORT.\n"
            "[11] FORCE_GLOBAL_ABORT: ABORT decision persisted.\n"
            "[12] SEND_ABORT: ABORT sent to Site S1.\n"
            "[13] SEND_ABORT: ABORT sent to Site S2.\n\n"
        )

    log.write("=" * 75 + "\n")
    log.write("FINAL TRANSACTION RESULT\n")
    log.write("=" * 75 + "\n\n")

    log.write("Global Decision: " + global_decision + "\n")

    log.write("Coordinator State: " + coordinator_state + "\n")

    for participant, state in participant_states.items():

        log.write(f"{participant}: {state}\n")

    log.write("\nIMPORTANT 2PC RULES:\n")

    log.write(
        "1. Participants must persist PREPARED before sending READY.\n"
    )

    log.write(
        "2. Coordinator must receive every participant's vote "
        "before deciding COMMIT or ABORT.\n"
    )

    log.write(
        "3. Any ABORT vote causes a global ABORT decision.\n"
    )

    log.write(
        "4. Coordinator persists its final decision before "
        "sending the decision message.\n"
    )

    log.write(
        "5. A participant in PREPARED state must not independently "
        "commit or abort merely because a message is delayed.\n"
    )

print("\nTransaction log created.")


# ================================================================
# STEP 7: CREATE SEQUENCE DIAGRAM (PNG)
# ================================================================

DIAGRAM_PATH = os.path.join(
    OUTPUT_DIR,
    "2PC_Sequence_Diagram.png"
)

fig, ax = plt.subplots(figsize=(16, 11))

ax.set_xlim(0, 12)
ax.set_ylim(0, 13)

ax.axis("off")

ax.set_title(
    "TWO-PHASE COMMIT PROTOCOL (2PC)\n"
    "Coordinator and Two Participant Sites",
    fontsize=18,
    fontweight="bold",
    pad=20
)

# Participant positions

x_positions = {
    "Coordinator": 2,
    "Site S1": 6,
    "Site S2": 10
}

names = [
    "Coordinator",
    "Site S1 - India",
    "Site S2 - Singapore"
]

for x, name in zip(
    [2, 6, 10],
    names
):

    ax.add_patch(
        Rectangle(
            (x - 1, 11.7),
            2,
            0.65,
            facecolor="#DBEAFE",
            edgecolor="#1D4ED8",
            linewidth=2
        )
    )

    ax.text(
        x,
        12.02,
        name,
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

    ax.plot(
        [x, x],
        [0.5, 11.7],
        linestyle="--",
        color="gray",
        linewidth=1.2
    )


def draw_message(
    x1,
    x2,
    y,
    label,
    color="#2563EB",
    style="-"
):

    arrow = FancyArrowPatch(
        (x1, y),
        (x2, y),
        arrowstyle="->",
        mutation_scale=15,
        linewidth=1.8,
        color=color,
        linestyle=style
    )

    ax.add_patch(arrow)

    ax.text(
        (x1 + x2) / 2,
        y + 0.15,
        label,
        ha="center",
        va="bottom",
        fontsize=9,
        color=color,
        fontweight="bold"
    )


# Phase 1: Prepare

ax.text(
    0.4,
    10.9,
    "PHASE 1: PREPARE",
    fontsize=12,
    fontweight="bold",
    color="#1D4ED8"
)

draw_message(2, 6, 10.3, "PREPARE")

draw_message(2, 10, 9.5, "PREPARE")

draw_message(
    6,
    2,
    8.7,
    "READY",
    color="#15803D"
)

draw_message(
    10,
    2,
    7.9,
    "READY",
    color="#15803D"
)

# Stable storage records

ax.text(
    6,
    7.25,
    "Site S1: FORCE PREPARED",
    ha="center",
    fontsize=8,
    color="#7C3AED"
)

ax.text(
    10,
    6.8,
    "Site S2: FORCE PREPARED",
    ha="center",
    fontsize=8,
    color="#7C3AED"
)

ax.text(
    2,
    7.2,
    "Coordinator receives all votes",
    ha="center",
    fontsize=9,
    color="#334155"
)

# Phase 2

ax.text(
    0.4,
    6.1,
    "PHASE 2: COMMIT",
    fontsize=12,
    fontweight="bold",
    color="#15803D"
)

draw_message(
    2,
    6,
    5.4,
    "COMMIT",
    color="#15803D"
)

draw_message(
    2,
    10,
    4.6,
    "COMMIT",
    color="#15803D"
)

# Final acknowledgements

draw_message(
    6,
    2,
    3.8,
    "ACKNOWLEDGEMENT",
    color="#475569"
)

draw_message(
    10,
    2,
    3.0,
    "ACKNOWLEDGEMENT",
    color="#475569"
)

# Failure recovery scenario

ax.text(
    0.4,
    2.0,
    "FAILURE RECOVERY SCENARIO",
    fontsize=11,
    fontweight="bold",
    color="#B45309"
)

ax.text(
    6,
    1.45,
    "If COMMIT message is missing:",
    ha="center",
    fontsize=9,
    color="#B45309"
)

ax.text(
    6,
    0.95,
    "Participant remains PREPARED and consults coordinator log.",
    ha="center",
    fontsize=9,
    color="#B45309"
)

ax.text(
    6,
    0.45,
    "It commits or aborts only after learning the global decision.",
    ha="center",
    fontsize=9,
    color="#B45309"
)

plt.tight_layout()

plt.savefig(
    DIAGRAM_PATH,
    dpi=250,
    bbox_inches="tight"
)

plt.close()

print("Sequence diagram PNG created.")


# ================================================================
# STEP 8: CONVERT SEQUENCE DIAGRAM INTO PDF
# ================================================================

DIAGRAM_PDF_PATH = os.path.join(
    OUTPUT_DIR,
    "2PC_Sequence_Diagram.pdf"
)

diagram_pdf = SimpleDocTemplate(
    DIAGRAM_PDF_PATH,
    pagesize=landscape(A4),
    rightMargin=20,
    leftMargin=20,
    topMargin=20,
    bottomMargin=20
)

styles = getSampleStyleSheet()

diagram_story = [

    Paragraph(
        "TWO-PHASE COMMIT SEQUENCE DIAGRAM",
        styles["Title"]
    ),

    Spacer(1, 15),

    PDFImage(
        DIAGRAM_PATH,
        width=760,
        height=510
    )

]

diagram_pdf.build(diagram_story)

print("Sequence diagram PDF created.")


# ================================================================
# STEP 9: CREATE RECOVERY PROCEDURE DOCUMENT (PDF)
# ================================================================

RECOVERY_PDF_PATH = os.path.join(
    OUTPUT_DIR,
    "2PC_Recovery_Procedure.pdf"
)

styles = getSampleStyleSheet()

styles.add(
    ParagraphStyle(
        name="RecoveryTitle",
        parent=styles["Title"],
        alignment=TA_CENTER,
        textColor=colors.HexColor("#1D4ED8"),
        fontSize=19
    )
)

recovery_story = []

recovery_story.append(
    Paragraph(
        "TWO-PHASE COMMIT RECOVERY PROCEDURE",
        styles["RecoveryTitle"]
    )
)

recovery_story.append(
    Paragraph(
        "Handling a Missing COMMIT Message After READY",
        styles["Heading2"]
    )
)

recovery_story.append(Spacer(1, 12))


recovery_sections = [

    (
        "1. Failure Scenario",
        """
        During a distributed transaction, Site S1 and Site S2
        receive PREPARE messages from the coordinator.

        Both participants write PREPARED records to stable storage
        before sending READY votes.

        The coordinator receives both READY votes, records the
        global COMMIT decision in its stable log, and sends COMMIT
        messages.

        However, Site S2 crashes or the network fails before
        Site S2 receives the COMMIT message.
        """
    ),

    (
        "2. Participant State After Failure",
        """
        Site S2 remains in the PREPARED state.

        Its transaction is considered an in-doubt transaction.

        Site S2 must retain the transaction information and
        required locks or other resources until the global
        outcome is known.

        It must not independently commit or abort the transaction
        simply because the COMMIT message is missing.
        """
    ),

    (
        "3. Recovery Procedure",
        """
        Step 1: Site S2 recovers its stable transaction log
        after restarting.

        Step 2: It identifies the transaction as PREPARED
        without a recorded final decision.

        Step 3: Site S2 contacts the coordinator and requests
        the transaction outcome.

        Step 4: The coordinator checks its stable log.

        Step 5: If the coordinator log contains GLOBAL_COMMIT,
        it retransmits COMMIT to Site S2.

        Step 6: Site S2 records COMMIT in stable storage,
        completes the transaction, and releases its resources.

        Step 7: If the coordinator log contains GLOBAL_ABORT,
        it retransmits ABORT. Site S2 records ABORT and
        releases its resources.
        """
    ),

    (
        "4. If the Coordinator Is Unavailable",
        """
        If Site S2 cannot contact the coordinator and no
        authoritative decision can be recovered, it must remain
        in the PREPARED state.

        This is called blocking in the Two-Phase Commit protocol.

        The participant waits until the coordinator or a
        reliable recovery mechanism provides the final decision.

        A participant must never guess the global outcome.
        """
    ),

    (
        "5. Coordinator Recovery",
        """
        After restarting, the coordinator reads its stable log.

        If a GLOBAL_COMMIT record exists, it retransmits COMMIT
        to every participant that has not acknowledged it.

        If a GLOBAL_ABORT record exists, it retransmits ABORT.

        If no final decision was recorded, the coordinator
        must follow the protocol's recovery rules and establish
        a safe global abort decision where permitted by the
        transaction state and protocol implementation.

        The coordinator records the decision durably before
        sending decision messages.
        """
    ),

    (
        "6. Conclusion",
        """
        Stable transaction logs make Two-Phase Commit recoverable.
        Participants persist PREPARED before voting READY, while
        the coordinator persists its final decision before
        broadcasting COMMIT or ABORT.

        This preserves atomicity: all participants eventually
        commit or all participants abort.
        """
    )

]


for heading, content in recovery_sections:

    recovery_story.append(
        Paragraph(
            heading,
            styles["Heading2"]
        )
    )

    recovery_story.append(
        Paragraph(
            content.replace("\n", "<br/>"),
            styles["BodyText"]
        )
    )

    recovery_story.append(
        Spacer(1, 12)
    )


# Recovery state table

recovery_story.append(
    Paragraph(
        "7. Recovery Decision Table",
        styles["Heading2"]
    )
)

recovery_table_data = [

    [
        "Coordinator Log",
        "Participant Action",
        "Final State"
    ],

    [
        "GLOBAL_COMMIT",
        "Receive or retransmit COMMIT",
        "COMMITTED"
    ],

    [
        "GLOBAL_ABORT",
        "Receive or retransmit ABORT",
        "ABORTED"
    ],

    [
        "No final decision / coordinator unavailable",
        "Wait and recover decision safely",
        "PREPARED / IN-DOUBT"
    ]

]

recovery_table = Table(
    recovery_table_data,
    colWidths=[165, 205, 110]
)

recovery_table.setStyle(
    TableStyle([
        (
            "BACKGROUND",
            (0, 0),
            (-1, 0),
            colors.HexColor("#DBEAFE")
        ),
        (
            "GRID",
            (0, 0),
            (-1, -1),
            0.7,
            colors.grey
        ),
        (
            "FONTNAME",
            (0, 0),
            (-1, 0),
            "Helvetica-Bold"
        ),
        (
            "FONTSIZE",
            (0, 0),
            (-1, -1),
            8
        ),
        (
            "VALIGN",
            (0, 0),
            (-1, -1),
            "MIDDLE"
        ),
        (
            "TOPPADDING",
            (0, 0),
            (-1, -1),
            8
        ),
        (
            "BOTTOMPADDING",
            (0, 0),
            (-1, -1),
            8
        )
    ])
)

recovery_story.append(recovery_table)


SimpleDocTemplate(
    RECOVERY_PDF_PATH,
    pagesize=A4,
    rightMargin=40,
    leftMargin=40,
    topMargin=40,
    bottomMargin=40
).build(recovery_story)

print("Recovery procedure PDF created.")


# ================================================================
# STEP 10: CREATE COMPLETE ASSIGNMENT ZIP
# ================================================================

ZIP_PATH = os.path.join(
    OUTPUT_DIR,
    "Task_2_Two_Phase_Commit.zip"
)

deliverables = [

    DIAGRAM_PATH,

    DIAGRAM_PDF_PATH,

    LOG_PATH,

    RECOVERY_PDF_PATH

]

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as zip_file:

    for filepath in deliverables:

        zip_file.write(
            filepath,
            arcname=os.path.basename(filepath)
        )


# ================================================================
# STEP 11: DISPLAY GENERATED FILES
# ================================================================

print("\n" + "=" * 65)

print("TASK 2 DELIVERABLES GENERATED SUCCESSFULLY")

print("=" * 65)

for filename in os.listdir(OUTPUT_DIR):

    filepath = os.path.join(
        OUTPUT_DIR,
        filename
    )

    size_kb = os.path.getsize(filepath) / 1024

    print(
        filename,
        " | ",
        round(size_kb, 2),
        "KB"
    )


# ================================================================
# STEP 12: AUTOMATICALLY DOWNLOAD ZIP
# ================================================================

print("\nDownloading Task 2 ZIP file...")

files.download(ZIP_PATH)

print("\nTASK 2 COMPLETED SUCCESSFULLY.")

Output directory created: /content/Task_2_Two_Phase_Commit

Transaction ID: TXN_2026_001
Coordinator: Coordinator
Participants: ['Site S1 - India', 'Site S2 - Singapore']

========== PHASE 1: PREPARE ==========

========== PHASE 2: DECISION ==========

========== FINAL TRANSACTION STATUS ==========
Global Decision: COMMIT
Coordinator State: COMMIT
Site S1 - India : COMMITTED
Site S2 - Singapore : COMMITTED

Transaction log created.
Sequence diagram PNG created.
Sequence diagram PDF created.
Recovery procedure PDF created.

TASK 2 DELIVERABLES GENERATED SUCCESSFULLY
2PC_Recovery_Procedure.pdf  |  4.19 KB
2PC_Sequence_Diagram.pdf  |  221.1 KB
2PC_Sequence_Diagram.png  |  233.5 KB
2PC_Transaction_Log.txt  |  2.08 KB
Task_2_Two_Phase_Commit.zip  |  336.12 KB



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


TASK 2 COMPLETED SUCCESSFULLY.
